# Test window-local heart-rate deviation and availability

Question: does an explicit past-relative heart-rate description improve stage detection with the same raw information window?
Reuse the original unsmoothed Adam/P3 reference on all five seed-42 development folds.
Train only five candidates; final-test observations remain reserved.

The input API remains thirty raw epochs with six prepared features. Inside the model, append two channels:
current heart-rate mean minus up to ten strictly previous means, and an availability flag requiring four past epochs.
The transform restarts at the beginning of EACH supplied window. It never reaches outside the existing window.

Session-precomputed deltas for the oldest tokens could extend effective context from thirty to forty epochs.
That would change the information boundary. The window-local implementation prevents this confound.
Its two channels enlarge the patch projection by 96 parameters: 2,756 becomes 2,852.
This is a feature-recipe comparison, not a parameter-matched proof.

Only available last-target deltas from training windows fit the extra normalization.
Unavailable standardized deltas are zero and their binary availability flag is zero. The flag remains unscaled.
Do not drop startup targets or use full-night normalization.
Read [the fixed protocol](../docs/window-local-heart-rate.md) before enabling preparation or training.

## Locate the checkout and declare execution switches

**What:** Find the repository, configure logging, and declare optional setup, preparation, checks, and training.

**Why:** Existing local and cloned checkouts need the same paths, while model fitting requires explicit opt-in.


In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("window_local_heart_rate_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/transformer-window-local-hr-grouped.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()

REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/window-local-heart-rate"  # Select the PR branch before merge
RUN_SETUP = False
RUN_PREPARATION = False
RUN_CHECK = True
RUN_TRAINING = False

logger.info("Repository: %s", ROOT)
logger.info("Setup=%s preparation=%s preflight=%s training=%s",
            RUN_SETUP, RUN_PREPARATION, RUN_CHECK, RUN_TRAINING)


## Optional environment setup

**What:** Optionally clone this working branch and install Astral uv plus locked dependency groups.

**Why:** A fresh machine needs project code and the same dependencies. The existing local environment can skip this cell.
Fresh clones also require development features and the completed five-fold reference artifacts.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")


## Verify the full-fold reference and fixed controls

**What:** Verify saved reference scores, participant assignments, measured recipes, device, and the one declared transform.

**Why:** The comparison must preserve raw inputs, target eligibility, the optimizer, loss, architecture settings, and training budget.
The only added model channels are the window-local heart-rate deviation and its availability flag.

In [ ]:
import json

import pandas as pd

from sleep_classifier.experiment_evidence import EpochEvidenceReport
from sleep_classifier.grouped_validation import GroupedValidationExperiment
from sleep_classifier.neural_training import TrainingConfig
from sleep_classifier.paired_architecture_report import PairedArchitectureReport

experiment = GroupedValidationExperiment(ROOT / "configs/transformer-window-local-hr-grouped.yaml")
reference = GroupedValidationExperiment(ROOT / "configs/transformer-patch-grouped.yaml")
plan = experiment.plan()
reference_summary = reference.output_dir / "summary.json"
if not reference_summary.exists():
    raise FileNotFoundError("The completed original Adam/P3 reference is required")

reference_saved = json.loads(reference_summary.read_text())
if reference_saved["plan"] != reference.plan():
    raise ValueError("The saved reference differs from the current participant-fold plan")
reference_validation = PairedArchitectureReport(reference_summary, reference_summary).build()
reference_manifest = json.loads((reference.output_dir / "manifest.json").read_text())
resolved_device = str(TrainingConfig(experiment.model_settings).device())
if resolved_device != reference_manifest["device"]:
    raise ValueError("Candidate device differs from the measured reference")
if plan["folds"] != reference_saved["plan"]["folds"]:
    raise ValueError("Candidate folds differ from the original participant assignments")
if len(plan["folds"]) != 5 or plan["maximum_epochs"] != 200:
    raise ValueError("The candidate must use five fits capped at 200 total epochs")
if len(plan["development"]) != 26 or len(plan["reserved_test"]) != 3:
    raise ValueError("Development or final-test membership differs from the frozen cohort")

excluded = {"window_features", "output_dir"}
recipe = {key: value for key, value in experiment.model_settings.items() if key not in excluded}
reference_recipe = {key: value for key, value in reference.model_settings.items()
                    if key not in excluded}
if recipe != reference_recipe:
    raise ValueError("Another model, optimization, feature, or diagnostic setting changed")
expected_window_features = {
    "name": "heart_rate_deviation", "history_epochs": 10, "min_history_epochs": 4,
}
if experiment.model_settings.get("window_features") != expected_window_features:
    raise ValueError("The candidate must use the declared window-local transform")
if reference.model_settings.get("window_features") is not None:
    raise ValueError("The reused reference must retain its original raw-six-channel path")
if (experiment.model_settings["context_epochs"] != 30
        or experiment.model_settings["patch_size"] != 3
        or experiment.model_settings["learning_rate"] != 0.001):
    raise ValueError("The raw context, patch size, and learning rate must remain fixed")
if any(experiment.model_settings.get(key) for key in (
    "optimizer", "optimizer_agent", "lr_scheduler", "loss",
)):
    raise ValueError("Keep original unsmoothed zero-decay Adam without a scheduler or agent")

artifact_fields = {"prepare_config", "output_dir", "device"}
expected_measured_recipe = {
    key: value for key, value in reference.model_settings.items() if key not in artifact_fields
}
for result in reference_saved["results"]:
    metrics_path = Path(result["metrics_path"])
    if not metrics_path.is_absolute():
        metrics_path = reference_summary.parent / metrics_path
    measured_config = json.loads(metrics_path.read_text())["config"]
    measured_recipe = {key: value for key, value in measured_config.items()
                       if key not in artifact_fields}
    if measured_recipe != expected_measured_recipe:
        raise ValueError("The current reference YAML differs from its measured fold recipe")

display(pd.DataFrame(plan["folds"]).set_index("name"))
display(pd.DataFrame([{
    "raw_context_epochs": 30, "raw_channels": 6, "effective_channels": 8,
    "patch_size_epochs": 3, "candidate_parameters": 2852, "new_fits": 5,
    "maximum_new_epochs": 200, "device": resolved_device,
}]))
logger.info("Reference verified; only window-local derived channels change")

## Inspect startup history and the derived-channel meaning

**What:** Calculate a small causal example before any fold preparation or training.

**Why:** The current epoch must stay outside its own past mean. Four prior epochs are required.
This table shows raw BPM deviation; preflight reports the fitted training-only normalization.
Availability remains a binary unscaled input. Missing standardized deviation is encoded as zero, with availability zero.
The same derivation runs independently inside each supplied raw window at training and inference.

In [ ]:
example_heart_rate = [60.0, 62.0, 64.0, 66.0, 70.0]
example_rows = []
for index, current in enumerate(example_heart_rate):
    previous = example_heart_rate[max(0, index - 10):index]
    available = len(previous) >= 4
    past_mean = sum(previous) / len(previous) if previous else None
    example_rows.append({
        "window_position": index + 1, "current_hr_bpm": current,
        "previous_epochs": len(previous), "strictly_past_mean_bpm": past_mean,
        "deviation_bpm": current - past_mean if available else None,
        "availability": int(available),
    })
display(pd.DataFrame(example_rows))
logger.info("The fifth position uses only the first four means: 70 - 63 = 7 BPM")

## Prepare only the window-local heart-rate candidate

**What:** Optionally freeze five candidate fold recipes and copy the existing unscaled feature tables.

**Why:** The raw-six-feature tables and eligible targets remain identical. Extra channels are derived inside each model window.
The candidate uses a separate output directory and never retrains the reference.

In [ ]:
if RUN_PREPARATION:
    PairedArchitectureReport(reference_summary, reference_summary).build()
    if json.loads(reference_summary.read_text())["plan"] != reference.plan():
        raise ValueError("The reference participant-fold plan changed")
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -m sleep_classifier.grouped_validation \
        --config configs/transformer-window-local-hr-grouped.yaml --prepare
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Preparation failed; inspect existing outputs before retrying")
else:
    logger.info("Patch candidate preparation skipped")

## Check every fold and inspect training-only normalization

**What:** Run five no-gradient checks and inspect raw/effective channels, fitted delta state, and model size.

**Why:** Each fold must retain raw input shape 64 × 30 × 6 and produce four logits.
The internal transform expands six channels to eight. Its normalization fits available training-target deltas only.
Validation targets and final-test observations must never fit normalization.

In [ ]:
if RUN_CHECK and (experiment.output_dir / "manifest.json").exists():
    %cd "{ROOT}"
    for fold in plan["folds"]:
        fold_config = experiment.output_dir / fold["name"] / "gru.yaml"
        logger.info("Checking %s without optimization", fold["name"])
        !uv run --locked --group gru --group notebook python -m sleep_classifier.neural_training \
            --config "{fold_config}" --check
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Patch candidate preflight failed")
elif RUN_CHECK:
    logger.info("Preflight pending: prepare the five folds first")
else:
    logger.info("Preflight skipped")

## Train five candidates when ready

**What:** Optionally train the five new window-local feature candidates with the original recipe.

**Why:** Training remains an explicit local action. The first-seed result must be evaluated before replication or promotion.
Keep the original Adam, unsmoothed loss, context thirty, rate 0.001, dropout 0.2, and budget forty/patience twelve.

In [ ]:
if RUN_TRAINING:
    PairedArchitectureReport(reference_summary, reference_summary).build()
    if json.loads(reference_summary.read_text())["plan"] != reference.plan():
        raise ValueError("The reference participant-fold plan changed")
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -u -m sleep_classifier.grouped_validation \
        --config configs/transformer-window-local-hr-grouped.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Patch training failed; inspect retained evidence before retrying")
else:
    logger.info("Training skipped: enable RUN_TRAINING when ready")

## Inspect completion status and the measured reference

**What:** Show available reference/candidate fold status and the reference's pooled selected-checkpoint scores.

**Why:** A complete reference does not imply a complete candidate. Pending runs must not become fabricated performance scores.


In [ ]:
reporters = {"reference": EpochEvidenceReport(reference.output_dir),
             "candidate": EpochEvidenceReport(experiment.output_dir)}
evidence_by_role = {role: reporter.build() for role, reporter in reporters.items()}
status_rows = [{"recipe": role, "study_status": evidence["status"], "fold": fold["fold"],
                "fold_status": fold["status"], "recorded_epochs": len(fold["epochs"])}
               for role, evidence in evidence_by_role.items() for fold in evidence["folds"]]
if status_rows:
    display(pd.DataFrame(status_rows).set_index(["recipe", "fold"]))
else:
    logger.info("The candidate has not been prepared")
display(pd.DataFrame([{"recipe": "measured reference",
                       **reference_validation["reference"]}]).drop(
    columns=["confusion_matrix", "classes"],
))

## Inspect available learning curves

**What:** Plot saved training/evaluation losses, macro-F1, gradients, and learning rates.

**Why:** The derived channels may affect both stage decisions and probability fit. A lower NLL alone is not a useful classification gain.
Both recipes retain ordinary hard-label cross-entropy and the same checkpoint-selection rule.

In [ ]:
import matplotlib.pyplot as plt

for role, reporter in reporters.items():
    if not any(fold["epochs"] for fold in evidence_by_role[role]["folds"]):
        logger.info("%s has no saved per-epoch evidence; no curve is reconstructed", role)
        continue
    figure = reporter.plot_curves()
    figure.suptitle(f"{role}: completed-epoch evidence")
    figure.tight_layout(rect=(0, 0, 1, 0.97))
    display(figure)
    plt.close(figure)

## Compare selected checkpoints after all five fits finish

**What:** Verify matched targets, effective transform provenance, fixed controls, and selected scores.

**Why:** The explicit window_features policy permits only this feature-recipe change, including its small projection-capacity difference.
The primary outcome remains paired mean fold macro-F1. Validation also selects checkpoints, so this is development evidence.

In [ ]:
comparison = None
candidate_summary = experiment.output_dir / "summary.json"
if candidate_summary.exists() and json.loads(candidate_summary.read_text())["status"] == "complete":
    comparison = PairedArchitectureReport(
        reference_summary, candidate_summary, comparison="window_features",
    ).build()
    display(pd.DataFrame(comparison["folds"]).set_index("fold"))
    score_rows = [{"recipe": role, "accuracy": comparison[role]["accuracy"],
                   "pooled_macro_f1": comparison[role]["macro_f1"]}
                  for role in ("reference", "candidate")]
    display(pd.DataFrame(score_rows).set_index("recipe"))
    class_rows = [{"recipe": role, "stage": stage, **scores}
                  for role in ("reference", "candidate")
                  for stage, scores in comparison[role]["classes"].items()]
    display(pd.DataFrame(class_rows).set_index(["recipe", "stage"]))
    destination = experiment.output_dir / "paired-comparison.json"
    destination.write_text(json.dumps(comparison, indent=2) + "\n")
    logger.info("Mean paired fold macro-F1 difference: %.6f",
                comparison["paired"]["mean_macro_f1_difference"])
else:
    logger.info("The patch candidate is incomplete; no paired conclusion is available")

## Visualize the pooled confusion matrices

**What:** Plot selected-checkpoint counts for true and predicted stages in the same order for both recipes.

**Why:** The matrices expose systematic stage substitutions that an aggregate score can hide.
Rows are true stages and columns are predicted stages. Raw counts reflect the cohort's class imbalance.


In [ ]:
if comparison is not None:
    import numpy as np
    from sklearn.metrics import ConfusionMatrixDisplay

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    for axis, role in zip(axes, ("reference", "candidate"), strict=True):
        ConfusionMatrixDisplay(
            np.asarray(comparison[role]["confusion_matrix"]),
            display_labels=["Wake", "Light", "Deep", "REM"],
        ).plot(ax=axis, colorbar=False, values_format="d")
        axis.set_title(role)
    figure.tight_layout()
    display(figure)
    plt.close(figure)
else:
    logger.info("Paired confusion matrices are pending candidate completion")

## Inspect candidate selected-checkpoint probability loss

**What:** Read selected-epoch validation NLL and evaluation-mode training NLL, then calculate pooled validation NLL.

**Why:** Negative log-likelihood measures probability assigned to the true stage; lower is better.
A confident wrong prediction receives a large penalty. NLL complements hard-label F1 and does not select our checkpoints.
Weight each fold by target count. Keep partial evidence separate from a complete pooled score.
Both displayed NLL measurements use unsmoothed hard targets. The smoothed training objective is not ordinary NLL.


In [ ]:
candidate_rows = []
selected_loss_sum = 0.0
selected_target_count = 0
for fold in evidence_by_role["candidate"]["folds"]:
    selected = next((epoch for epoch in fold["epochs"]
                     if epoch["epoch"] == fold["best_epoch"]), None)
    if selected is None:
        continue
    count = sum(stage["count"] for stage in selected["validation_classes"].values())
    selected_target_count += count
    selected_loss_sum += selected["validation_nll"] * count
    candidate_rows.append({"fold": fold["fold"], "selected_epoch": fold["best_epoch"],
                           "selected_validation_nll": selected["validation_nll"],
                           "selected_training_evaluation_nll": selected["training_evaluation_nll"],
                           "scored_targets": count})
if candidate_rows:
    display(pd.DataFrame(candidate_rows).set_index("fold"))
    if len(candidate_rows) == 5:
        logger.info("Candidate pooled selected-checkpoint NLL: %.6f",
                    selected_loss_sum / selected_target_count)
    else:
        logger.info("Partial fold NLL only; no complete candidate aggregate yet")
else:
    logger.info("Selected-checkpoint NLL is pending candidate completion")

## Compare startup and transition slices

**What:** Compare paired scores by available raw history and true-stage transitions after training completes.

**Why:** Startup targets remain eligible despite unavailable derived history. Inspect supports and REM/Deep trade-offs beside macro-F1.
These post-hoc labels define evaluation slices only. They never become model inputs.
Use history bins 1–4, 5–29, and full context. The first bin has unavailable target-level history.
These bins retain all 22,708 eligible targets. Do not average their macro-F1 scores to reconstruct pooled macro-F1.


In [ ]:
from sleep_classifier.temporal_slice_report import TemporalSliceReport

if comparison is not None:
    temporal_report = TemporalSliceReport.from_grouped(
        reference_summary, candidate_summary, short_history_epochs=4,
        comparison="window_features",
    ).build()
    slice_rows = []
    slice_class_rows = []
    for row in temporal_report["slices"]:
        for role in ("reference", "candidate"):
            scores = row[role]
            if scores is None:
                continue
            slice_rows.append({"slice": row["name"], "recipe": role,
                               "targets": row["targets"], "accuracy": scores["accuracy"],
                               "macro_f1": scores["macro_f1"]})
            slice_class_rows.extend(
                {"slice": row["name"], "recipe": role, "stage": stage, **values}
                for stage, values in scores["classes"].items()
            )
    display(pd.DataFrame(slice_rows).set_index(["slice", "recipe"]))
    display(pd.DataFrame(slice_class_rows).set_index(["slice", "recipe", "stage"]))
    (experiment.output_dir / "temporal-slices.json").write_text(
        json.dumps(temporal_report, indent=2) + "\n",
    )
else:
    logger.info("Temporal slices require five complete paired candidate fits")

## Save the local evidence summary

**What:** Regenerate the candidate's portable local evidence report without loading or changing its model.

**Why:** The same inspector supports prepared, partial, failed, and completed studies.
An unprepared report does not create a directory that would block later preparation.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python -m sleep_classifier.experiment_evidence \
    --output-dir artifacts/transformer-window-local-hr-grouped-validation --report
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Patch evidence inspection failed")

## Export the executed HTML report

**What:** Execute report and preflight cells through a fresh kernel, then save local HTML.

**Why:** The source notebook remains output-free, while a reproducible executed report preserves inspection results.
Automated export skips setup, preparation, training, and this recursive export cell regardless of execution switches.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/29_window_local_heart_rate.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Patch notebook export failed")

## Measured seed-42 findings

Mean fold macro-F1 rises from 0.56538 to 0.57854. Four of five folds improve.
Pooled accuracy rises from 64.15% to 65.96%; pooled REM F1 rises from 0.49905 to 0.54925.
Fold 01 provides the largest gain. Wake F1 falls, Deep recall falls, and transition scores worsen.
Selected pooled hard-label NLL falls from 0.89553 to 0.85473, but late training still shows probability overfitting.
These are promising one-seed development results, not final-test evidence or isolated proof of deviation value.
Repeat the unchanged candidate with seeds 43 and 44 before promoting it. Reuse their existing original-P3 reference results.
Read the [measured findings](../docs/experiments/window-local-heart-rate-2026-11-17/README.md) for fold, class, participant, and temporal evidence.

The model input contract is the data passed to the Python model, not an HTTP endpoint.
Callers still supply the original six prepared features and valid lengths; the model derives the two added channels internally.
